# Label transfer (Macaque snRNA to ST)

In [1]:
"""
Refactored version:
- All paths and parameters are configured via command-line arguments.
- Fixed the evaluation logic to compute and save metrics.
- Removed hardcoded paths and added instructions for dependency management.
- Improved code structure and comments for better readability and portability.
- Added and fixed the subsampling logic.
- Corrected the preprocessing order: smooth first, then downsample.
"""
import os
import torch
import scanpy as sc
import matplotlib.pyplot as plt
import argparse
import warnings
import itertools
from sklearn.metrics import classification_report, adjusted_rand_score, accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
from anndata import AnnData
from typing import Literal, Dict, List, Tuple, Optional

import sys
sys.path.append("/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/07_refine/BrainBeacon-main")

from model.pipeline.label_transfer_utils import _subsample_adata, _preprocess_one_adata, plot_spatial_comparison
from model.pipeline.cell_label_transfer import run_prediction_pipeline

# --- Dependency Notes ---
# 1. BrainBeacon Project:
# This script depends on the 'BrainBeacon-main' project. Please ensure this project is
# installed as a Python package (e.g., via `pip install -e .` in its directory) or
# that its path is added to your PYTHONPATH environment variable.
#
# 2. Label Transfer Functions:
# This script requires a file named 'label_trasfer_function_9001.py' containing the
# `run_label_transfer` function. Make sure this file is in the same directory as this
# script or accessible in your PYTHONPATH.

from config.config_cdniche import GENE_DICT_PATH
from config.config_train_cdniche import config_train
from model.utils import get_gene_mean_path, spatial_expression_imputation, ensure_ensembl_ids, set_seed
from model.pipeline.cell_embedding import run_bbcellformer_pipeline
from model.pipeline.cell_label_transfer import train_encoder_on_multi_adata
from model.pipeline.label_transfer_utils import *

# Fixed configuration, no need to modify
 # Set GPU
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

Using device: cuda
Using GPU: NVIDIA A100-SXM4-80GB


In [2]:
args = argparse.Namespace(
    # --- Dataset & Path Parameters ---
    asset_dir="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/model_path/BrainBeacon-main/",  # BrainBeacon-main BASE DIR
    aux_data_dir="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/data/Cortex",
    ref_path="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/data/Cortex/macaque1_snRNA_filtered1000.h5ad",
    ref_name="macaque_snRNA",
    ref_specie="macaque",
    ref_assay="snrna",
    query_path="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/data/Cortex/T33_raw_counts.h5ad",
    query_name="macaque_T33",
    query_specie="macaque",
    query_assay="stereo",
    output_dir="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1",
    bb_ckpt_name="epoch_0_step_800000_0.33B.pt",

    # --- Preprocessing Parameters ---
    n_hvg=3000,
    convert_id=False,  # whether to convert gene IDs to reference species
    smooth_ref=False,  # whether to smooth reference data
    smooth_k_ref=25,
    smooth_query=True,
    smooth_k_query=25,
    add_genes=False,  # whether to add genes from reference prior to query
    align_genes=False,  # whether to align genes between ref and query by intersection
    # --- Subsampling Parameters ---
    sample_mode="fix",
    # sample_mode="prop",  # "fix", "prop", "none"
    min_cells_per_class=100,
    sample_rate=0.05,
    alpha=0.2,
    marker_csv="/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/macaque_snRNA.marker.subclass.csv",
    # marker_csv=None,
    marker_topn=50,  # can be None
    marker_cutoff=1.0,     # cutoff for marker logFC
    pool_factor=1,
    oversample_dict=None,
    # oversample_dict={"VIP": 2, "VIP_RELN": 2},
    max_query_cells=None,  # max cells to use in query for test
    # max_query_cells=10000,  # max cells to use in query for test

    # --- Training & Evaluation Parameters ---
    n_hvg_train=1000,
    num_global_epochs=10,
    per_dataset_epochs=10,
    batch_size=32,
    class_col="SubClass",
    true_label_col='SubClass',
    # class_col="Class",
    emb_keys=["X_emb"],
    norm_types=["raw"],
    methods=["native",],  # can be ["native", "hnsw", "faiss", "prototype", "logreg"]
    k_values=[10],  # k for knn-based methods
    metrics=["cosine"],
    unassigned_threshold=None,
    force_preprocess=False,
    # skip_training=False,
    skip_training=True,
    skip_evaluation=False,
    eval_mode="benchmark",  # "spatial" or "benchmark"
)

In [3]:
# def main(args=None):
"""Main execution pipeline."""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

# --- Setup Datasets and Paths ---
raw_dataset_info_list = [
    {'data_name': args.ref_name, 'adata_path': args.ref_path, 'specie': args.ref_specie, 'assay': args.ref_assay},
    {'data_name': args.query_name, 'adata_path': args.query_path, 'specie': args.query_specie, 'assay': args.query_assay}
]

ref_info = raw_dataset_info_list[0]
query_info = raw_dataset_info_list[1]
print(f"[INFO] Reference dataset: {ref_info['data_name']} (Assay: {ref_info['assay']}) at {ref_info['adata_path']}")
print(f"[INFO] Query dataset: {query_info['data_name']} (Assay: {query_info['assay']}) at {query_info['adata_path']}")

# --- Load marker dict if provided ---
marker_dict = None
if args.marker_csv is not None:
    marker_dict = build_marker_dict(
        args.marker_csv, class_col=args.class_col,
        cutoff=args.marker_cutoff, top_n=args.marker_topn,
    )
    print(f"[INFO] Loaded marker dict with {len(marker_dict)} classes from {args.marker_csv}")

add_genes_list = None
# from predefined subclass markers
subclass_marker_genes_dict = {
    "EX": ["SLC17A6"], "L2": ["CCBE1", "CUX2"], "L2/3": ["GPC5", "PDZD2"],
    "L4": ["MYLK", "PLCH1", "RORB"], "L4/5": ["IL1RAPL2"],
    "L5/6": ["ETV1", "TLE4", "SEMA3E", "FOXP2"], "IN": ["GAD1", "GAD2"],
    "LAMP5": ["LAMP5"], "RELN": ["RELN"], "VIP": ["VIP"], "PV_chc": ["PVALB", "ADAMTSL1"],
    "PVALB": ["PVALB"], "SST": ["SST"], "ASC": ["SLC1A2", "SLC1A3"],
    "OPC": ["PTPRZ1", "PDGFRA"], "OLG": ["PLP1"], "MG": ["ITGAM"],
    "EC": ["RGS5"], "VLMC": ["COL1A2"]
}
if args.add_genes:
    extra_genes_marker = set()
    extra_genes_subclass = set()

    # from marker_dict
    if marker_dict is not None:
        for genes in marker_dict.values():
            extra_genes_marker.update(genes.keys())

    # from predefined subclass markers
    for genes in subclass_marker_genes_dict.values():
        extra_genes_subclass.update(genes)

    # union
    add_genes_list = list(extra_genes_marker.union(extra_genes_subclass))

    print(f"[INFO] Extra genes from marker_dict: {len(extra_genes_marker)}")
    print(f"[INFO] Extra genes from subclass dict: {len(extra_genes_subclass)}")
    print(f"[INFO] Collected {len(add_genes_list)} unique extra genes for HVG inclusion")

Using device: cuda
Using GPU: NVIDIA A100-SXM4-80GB
[INFO] Reference dataset: macaque_snRNA (Assay: snrna) at /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/data/Cortex/macaque1_snRNA_filtered1000.h5ad
[INFO] Query dataset: macaque_T33 (Assay: stereo) at /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/data/Cortex/T33_raw_counts.h5ad
[INFO] Marker gene stats: min=50, max=50, mean=50.0, total_classes=22
[INFO] Loaded marker dict with 22 classes from /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/macaque_snRNA.marker.subclass.csv


In [4]:
# === Sampling config ===
parts = [f"{args.sample_mode}"]
if args.sample_mode == "fix":
    parts.append(f"{args.min_cells_per_class}")
elif args.sample_mode == "prop":
    parts.append(f"{int(args.sample_rate * 100)}pct_alpha{args.alpha}")
parts.append(args.class_col)

if marker_dict is not None:
    parts.append(f"marker{args.marker_topn}")
    if args.pool_factor != 1:
        parts.append(f"pool{args.pool_factor}")

if args.oversample_dict:
    oversample_str = "over" + "_".join([f"{k}{v}" for k, v in args.oversample_dict.items()])
    parts.append(oversample_str)

if args.max_query_cells is not None:
    parts.append(f"qSub{args.max_query_cells}")

# === Preprocessing config ===
if args.convert_id and ref_info["specie"] != query_info["specie"]:
    parts.append("convID")
if args.smooth_ref:
    parts.append(f"sRef{args.smooth_k_ref}")
if args.smooth_query:
    parts.append(f"sQuery{args.smooth_k_query}")

# === Gene config ===
parts.append(f"hvg{args.n_hvg}")
if args.add_genes:
    parts.append("addGenes")
if args.align_genes:
    parts.append("gAlign")

# --- Finalize suffix ---
suffix = "_".join(parts)
args.output_dir = os.path.join(args.output_dir, suffix)
os.makedirs(args.output_dir, exist_ok=True)

In [5]:
# ===== Step 1: Preprocessing =====
processed_paths = {
    ref_info["data_name"]: os.path.join(args.output_dir, f"{ref_info['data_name']}_hvg{args.n_hvg}.h5ad"),
    query_info["data_name"]: os.path.join(args.output_dir, f"{query_info['data_name']}_hvg{args.n_hvg}.h5ad"),
}

adata_dict = {}
if not args.force_preprocess and all(os.path.exists(p) for p in processed_paths.values()):
    print("[INFO] Using cached processed files.")
    for name, path in processed_paths.items():
        adata_dict[name] = sc.read_h5ad(path)
else:
    print("[INFO] Processed files not found or --force-preprocess is set, running preprocessing...")
    adata_ref = sc.read_h5ad(ref_info["adata_path"])
    adata_query = sc.read_h5ad(query_info["adata_path"])
    print(f"[INFO] ref loaded {adata_ref.shape}, query loaded {adata_query.shape}")

    # --- Preprocessing (Corrected Order) ---
    adata_ref = _preprocess_one_adata(
        adata_ref, ref_info, args.n_hvg,
        target_species=ref_info["specie"],
        convert_id=False,
        smooth_st=args.smooth_ref,
        smooth_k=args.smooth_k_ref,
        add_genes=add_genes_list
    )
    adata_query = _preprocess_one_adata(
        adata_query, query_info, args.n_hvg,
        target_species=ref_info["specie"],
        convert_id=args.convert_id,
        smooth_st=args.smooth_query,
        smooth_k=args.smooth_k_query,
        add_genes=add_genes_list
    )
    print(f"[INFO] After preprocessing: ref {adata_ref.shape}, query {adata_query.shape}")

    rename_map = {"class": "Class", "subclass": "SubClass"}
    adata_ref.obs.rename(columns={k: v for k, v in rename_map.items() if k in adata_ref.obs}, inplace=True)
    adata_query.obs.rename(columns={k: v for k, v in rename_map.items() if k in adata_query.obs}, inplace=True)
    if args.class_col == "Class":
        new_class = np.where(
            adata_ref.obs["Class"] == "NonNeuron",  # 改这里
            adata_ref.obs["SubClass"],
            adata_ref.obs["Class"]
        )
        adata_ref.obs["Class"] = pd.Categorical(new_class)

    # --- Align genes by intersection ---
    if args.align_genes:
        common_genes = adata_ref.var_names.intersection(adata_query.var_names)
        print(f"[INFO] Aligning genes: {len(common_genes)} common genes found")
        adata_ref = adata_ref[:, common_genes].copy()
        adata_query = adata_query[:, common_genes].copy()
    else:
        print("[INFO] Gene alignment skipped.")

    # --- Subsampling (Corrected Placement) ---
    if args.sample_mode != "none":
        if args.class_col not in adata_ref.obs:
            warnings.warn(f"Cannot subsample: class column '{args.class_col}' not in reference data. Skipping subsampling.")
        else:
            # adata_ref = _subsample_adata(adata_ref, args.class_col, args.sample_mode, args.min_cells_per_class, args.sample_rate)
            adata_ref = _subsample_adata(
                adata_ref, args.class_col, args.sample_mode,
                args.min_cells_per_class, args.sample_rate, args.alpha,
                marker_dict=marker_dict, pool_factor=args.pool_factor,
                oversample_dict=args.oversample_dict,
            )
    # --- Optional random subsampling for query (label-free) ---
    if args.max_query_cells is not None and adata_query.n_obs > args.max_query_cells:
        sampled_idx = np.random.choice(adata_query.n_obs, args.max_query_cells, replace=False)
        adata_query = adata_query[sampled_idx, :].copy()
        print(f"[INFO] Query subsampled to {adata_query.n_obs} cells (random, label-free).")

    # --- Finalize for saving ---
    adata_dict[ref_info["data_name"]] = adata_ref
    adata_dict[query_info["data_name"]] = adata_query

    for name, adata in adata_dict.items():
        save_path = processed_paths[name]
        adata.obs.index.name = None
        adata.var_names.index.name = None
        # adata.obsm["spatial"] = np.asarray(adata.obsm["spatial"], dtype=float)
        adata.write_h5ad(save_path)
        print(f"[INFO] Saved {name} to {save_path}")

[INFO] Using cached processed files.


In [9]:
# ===== Step 2: Model Fine-tuning =====
method_name = f"brainbeacon_hvg{args.n_hvg_train}_total{args.num_global_epochs}_fit{args.per_dataset_epochs}"
final_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"flowformer_epoch{args.num_global_epochs}.pt")

if args.skip_training and os.path.exists(final_ckpt_path):
    print(f"[INFO] Skipping training or using existing checkpoint: {final_ckpt_path}")
else:
    print(f"[INFO] Fine-tuning encoder. Final checkpoint will be at {final_ckpt_path}")
    dataset_info_list_train = [
        {"data_name": ds["data_name"], "data_dir": args.output_dir, "adata_name": os.path.basename(processed_paths[ds["data_name"]]),
         "specie": ds["specie"], "assay": ds["assay"]}
        for ds in raw_dataset_info_list
    ]
    final_ckpt_path = train_encoder_on_multi_adata(
        dataset_info_list=dataset_info_list_train,
        bb_ckpt_path=os.path.join(args.asset_dir, "pretrained", args.bb_ckpt_name),
        # initial_ckpt_path=os.path.join(args.asset_dir, "pretrained", "epoch_0_step_800000_0.33B", "cellformer_epoch99.pt"),
        initial_ckpt_path=os.path.join(args.asset_dir, "pretrained", "cellformer_epoch99.pt"),
        output_dir=args.output_dir,
        config_train=config_train, output_prefix=method_name,
        num_global_epochs=args.num_global_epochs, per_dataset_epochs=args.per_dataset_epochs,
        n_hvg=args.n_hvg_train, batch_size=args.batch_size, enc_mod="flowformer", device=device
    )

[INFO] Fine-tuning encoder. Final checkpoint will be at /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/checkpoints/flowformer_epoch10.pt

========== Global Epoch 1 ==========

--- Training on macaque_T33 ---
Tokenized data found (40 .parquet, 40 dirs). Skipping tokenization.
Found existing BB embeddings at: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/macaque_T33/brainbeacon_hvg1000_total10_fit10_bb_embeddings.npz
[INFO] Using explicitly provided CellFormer checkpoint: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/model_path/BrainBeacon-main/pretra

/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/07_refine/BrainBeacon-main/model/bbcellformer/utils/data.py:20: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  return torch.sparse_csr_tensor(x.indptr, x.indices, x.data, (x.shape[0], x.shape[1])).to_sparse().float().coalesce()
 10%|█         | 1/10 [00:00<00:03,  2.30it/s]

Epoch 0 | Train loss: 768.1413 | Valid loss: 772.2471



 20%|██        | 2/10 [00:00<00:02,  3.06it/s]

Epoch 1 | Train loss: 766.7123 | Valid loss: 769.8161


...

 80%|████████  | 8/10 [00:01<00:00,  6.31it/s]

Epoch 6 | Train loss: 2799.9211 | Valid loss: 2827.8945
Epoch 7 | Train loss: 2785.7483 | Valid loss: 2829.9253


100%|██████████| 10/10 [00:01<00:00,  6.30it/s]

Epoch 8 | Train loss: 2775.3994 | Valid loss: 2834.5503
Epoch 9 | Train loss: 2767.7144 | Valid loss: 2819.3601


Model saved to /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/checkpoints/flowformer_epoch10.pt
Finished training on macaque_snRNA, model_raw saved to: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/checkpoints/flowformer_epoch10.pt


In [10]:
# ===== Step 3: Inference =====
print("[INFO] Running inference to generate embeddings...")
for ds in raw_dataset_info_list:
    data_name = ds["data_name"]
    adata_path = processed_paths[data_name]
    output_dir_final = os.path.join(args.output_dir, data_name)

    adata = run_bbcellformer_pipeline(
        adata_path=adata_path, specie=ds["specie"], assay=ds["assay"],
        gene_dict_path=GENE_DICT_PATH,
        gene_mean_path=get_gene_mean_path(args.asset_dir, ds["assay"], use_metacell=True),
        bb_ckpt_path=os.path.join(args.asset_dir, "pretrained", args.bb_ckpt_name),
        cellplm_ckpt_path=final_ckpt_path,
        output_dir=output_dir_final, output_prefix=method_name,
        config_train=config_train, n_hvg=args.n_hvg_train,
        force_tokenize=False, do_fit=False, # Inference only
        enc_mod="flowformer", device=device, seed=42
    )
    adata_dict[data_name] = adata
    print(f"Inference completed for {data_name}.")

[INFO] Running inference to generate embeddings...
Tokenized data found (1 .parquet, 1 dirs). Skipping tokenization.
Skipping BB inference. Found existing file: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/macaque_snRNA/brainbeacon_hvg1000_total10_fit10_bb_embeddings.npz
[INFO] Using explicitly provided CellFormer checkpoint: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/checkpoints/flowformer_epoch10.pt
********** gene list size: 92076 **********
********** loading skip parameters: set() **********
After filtering, 3000 genes remain.
Model saved to /cluster/home/yangyiwen/work_file/Brainbeacon

In [11]:
# ===== Step 4: Evaluation =====
adata_ref = adata_dict[ref_info['data_name']].copy()
adata_query = adata_dict[query_info['data_name']].copy()
results = []

if not args.skip_evaluation:
    print("[INFO] Starting evaluation of different label transfer methods...")
    param_combinations = list(itertools.product(args.emb_keys, args.norm_types, args.methods))

    for emb_key, norm_type, method in param_combinations:
        if emb_key not in adata_ref.obsm or emb_key not in adata_query.obsm:
            warnings.warn(f"Embedding key '{emb_key}' not found. Skipping combination.")
            continue

        X_ref = adata_ref.obsm[emb_key]
        X_query = adata_query.obsm[emb_key]

        if args.class_col not in adata_ref.obs:
            raise ValueError(f"Reference annotation column '{args.class_col}' not found in reference adata.")
        y_ref = adata_ref.obs[args.class_col].astype(str)

        if norm_type != "raw":
            X_ref = normalize(X_ref, norm=norm_type)
            X_query = normalize(X_query, norm=norm_type)

        grid = []
        if method in ["native", "hnsw", "faiss"]:
            grid = itertools.product(args.k_values, args.metrics)
        elif method == "prototype":
            grid = [(None, m) for m in args.metrics]
        else: # logreg
            grid = [(None, None)]

        for K, metric in grid:
            print(f"\n=== Evaluating: {emb_key} | {norm_type} | {method} | K={K} | metric={metric} ===")

            # --- get threshold from args ---
            ua_thr = getattr(args, "unassigned_threshold", None)
            if ua_thr is not None:
                print(f"Using Unassigned threshold = {ua_thr}")
            y_query_pred = run_label_transfer(
                X_ref, y_ref, X_query, method=method,
                K=K if K else 30, metric=metric if metric else "euclidean",
                weight_mode="distance",
                unassigned_threshold=ua_thr,
                device=device,
            )

            pred_key = f"pred_{emb_key}_{norm_type}_{method}"
            if K: pred_key += f"_k{K}"
            if metric: pred_key += f"_{metric}"
            if ua_thr is not None:
                pred_key += f"_thr{ua_thr}"
            adata_query.obs[pred_key] = pd.Categorical(y_query_pred)

            if args.eval_mode == "spatial":
                spatial_plot_dir = os.path.join(args.output_dir, "spatial_plots")
                os.makedirs(spatial_plot_dir, exist_ok=True)
                output_path = os.path.join(spatial_plot_dir, f"{pred_key}_spatial.png")
                # store suffix and query name for plot title
                adata_query.uns["suffix"] = suffix
                adata_query.uns["query_name"] = query_info["data_name"]

                plot_spatial_comparison(
                    adata=adata_query,
                    true_label_col=args.true_label_col,
                    pred_label_col=pred_key,
                    output_path=output_path,
                    exclude_unassigned=False,
                )
            elif args.eval_mode == "benchmark":
                raw_data_path = args.aux_data_dir
                pretrained_model = pd.read_csv(
                    os.path.join(raw_data_path, "metaneighbor_pretrained_scrna_sct_classII.txt"),
                    sep=" ", index_col=0)
                result_dir = os.path.join(args.output_dir, pred_key)
                eval_result = run_prediction_pipeline(
                    adata=adata_query,
                    pretrained_model=pretrained_model,
                    marker_gene_dict=subclass_marker_genes_dict,
                    output_folder=result_dir,
                    true_label_col=args.class_col,
                    pred_col_name=pred_key,
                )

if results:
    results_df = pd.DataFrame(results)
    summary_path = os.path.join(args.output_dir, "results_summary.csv")
    results_df.to_csv(summary_path, index=False)
    print(f"\n[INFO] Results summary saved to {summary_path}")
    print("--- Evaluation Summary ---")
    print(results_df)

[INFO] Starting evaluation of different label transfer methods...

=== Evaluating: X_emb | raw | native | K=10 | metric=cosine ===
--- Pipeline started. Output will be saved to: /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/pred_X_emb_raw_native_k10_cosine ---
--- 1. Running MetaNeighborUS to get predictions ---


/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/pymn/utils.py:98: UserWarning: Replacing any | with a . in study column values
  warnings.warn("Replacing any | with a . in study column values")


Predictions stored in 'adata.obs[pred_X_emb_raw_native_k10_cosine]'.


/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/pymn/plotting.py:127: RuntimeWarning: invalid value encountered in divide
  row_score = np.nansum(M2.T * np.arange(M2.shape[1])[:, None],
/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/seaborn/matrix.py:1124: UserWarning: ``square=True`` ignored in clustermap
  warnings.warn(msg)



--- 2. Starting evaluation and visualization ---
Saving marker gene dotplot...
[WARN] 2 marker genes not found in adata, filtering...
Analyzing distribution in layers...
Comparing cell type proportions...
Generating confusion matrices...
Generating spatial plots...


/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/07_refine/BrainBeacon-main/model/pipeline/cell_label_transfer.py:736: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/07_refine/BrainBeacon-main/model/pipeline/cell_label_transfer.py:746: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


Calculating classification metrics...


/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/cluster/home/yangyiwen/anaconda3/envs/brainbeacon_dgx/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this 


--- Pipeline finished successfully. All outputs are in /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/pred_X_emb_raw_native_k10_cosine ---


In [12]:
# ===== Step 5: Save Final AnnData Objects =====
print("[INFO] Saving final AnnData objects with embeddings and predictions.")
save_query_path = os.path.join(args.output_dir, f"{query_info['data_name']}_with_predictions.h5ad")
adata_query.write_h5ad(save_query_path)
print(f"[INFO] Saved query adata to {save_query_path}")

save_ref_path = os.path.join(args.output_dir, f"{ref_info['data_name']}_with_embeddings.h5ad")
adata_ref.write_h5ad(save_ref_path)
print(f"[INFO] Saved reference adata to {save_ref_path}")

[INFO] Saving final AnnData objects with embeddings and predictions.
[INFO] Saved query adata to /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/macaque_T33_with_predictions.h5ad
[INFO] Saved reference adata to /cluster/home/yangyiwen/work_file/Brainbeacon_project/yangqq_workdir/06_cross_species_pipline/cross_species_zcm/analysis_result/downstream_tasks/cell_label_transfer/1011_out/macaqueSN_to_macaqueT33_jt1/fix_100_SubClass_marker50_sQuery25_hvg3000/macaque_snRNA_with_embeddings.h5ad


In [ ]:
if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1:  # 有命令行参数
        parser = argparse.ArgumentParser(description="Cross-species Label Transfer Pipeline using BrainBeacon")

        # --- Paths and Directories ---
        parser.add_argument('--asset_dir', type=str, required=True, help="Path to BrainBeacon assets (pretrained models, etc.)")
        parser.add_argument('--output_dir', type=str, required=True, help="Directory to save outputs")
        parser.add_argument('--aux_data_dir', type=str, help="Path to auxiliary data like metaneighbor files")

        # --- Reference Dataset ---
        parser.add_argument('--ref_path', type=str, required=True, help="Path to reference .h5ad file")
        parser.add_argument('--ref_name', type=str, required=True, help="Name for the reference dataset (e.g., 'macaque_T33')")
        parser.add_argument('--ref_specie', type=str, required=True, choices=['macaque', 'marmoset', 'human', 'mouse'], help="Reference species")
        parser.add_argument('--ref_assay', type=str, required=True, choices=['stereo', 'snrna'], help="Reference assay type")

        # --- Query Dataset ---
        parser.add_argument('--query_path', type=str, required=True, help="Path to query .h5ad file")
        parser.add_argument('--query_name', type=str, required=True, help="Name for the query dataset (e.g., 'marmoset_T454')")
        parser.add_argument('--query_specie', type=str, required=True, choices=['macaque', 'marmoset', 'human', 'mouse'], help="Query species")
        parser.add_argument('--query_assay', type=str, required=True, choices=['stereo', 'snrna'], help="Query assay type")

        # --- Preprocessing Parameters ---
        parser.add_argument('--n_hvg', type=int, default=6000, help="Number of highly variable genes for preprocessing")
        parser.add_argument('--convert_id', action='store_true', help="Convert gene IDs to reference species")
        parser.add_argument('--smooth_ref', action='store_true', help="Apply spatial smoothing to reference data")
        parser.add_argument('--smooth_k_ref', type=int, default=25, help="Number of neighbors for smoothing reference data")
        parser.add_argument('--smooth_query', action='store_true', help="Apply spatial smoothing to query data")
        parser.add_argument('--smooth_k_query', type=int, default=25, help="Number of neighbors for smoothing query data")
        parser.add_argument('--add_genes', action='store_true', help="Add genes from reference prior to query")
        parser.add_argument('--align_genes', action='store_true', help="Align genes between reference and query by intersection")

        # --- Subsampling Parameters ---
        parser.add_argument('--sample_mode', type=str, default="none", choices=['none', 'fix', 'prop'], help="Subsampling mode for reference data ('none', 'fix', 'prop')")
        parser.add_argument('--min_cells_per_class', type=int, default=100, help="Max number of cells per class for 'fix' mode")
        parser.add_argument('--sample_rate', type=float, default=0.1, help="Fraction of cells to keep for 'prop' mode")
        parser.add_argument('--marker_csv', type=str, default=None, help="Path to marker CSV file (optional). If provided, marker scores will guide subsampling.")
        parser.add_argument('--marker_topn', type=int, default=100, help="Top N marker genes to consider per class")
        parser.add_argument('--marker_cutoff', type=float, default=0.25, help="Cutoff for marker logFC")
        parser.add_argument('--pool_factor', type=int, default=1, help="Pool factor for marker-based subsampling")
        parser.add_argument("--oversample_dict", type=str, default=None, help="Dictionary for oversampling specific classes, e.g., '{\"VIP\": 2, \"VIP_RELN\": 2}'")
        parser.add_argument('--max_query_cells', type=int, default=None, help="If set, randomly subsample query cells to this number (label-free).")

        # --- Model and Training Parameters ---
        parser.add_argument('--bb_ckpt_name', type=str, default="epoch_0_step_800000_0.33B.pt", help="BrainBeacon checkpoint file name")
        parser.add_argument('--n_hvg_train', type=int, default=1000, help="Number of HVGs for model_raw training")
        parser.add_argument('--num_global_epochs', type=int, default=100, help="Number of global training epochs")
        parser.add_argument('--per_dataset_epochs', type=int, default=50, help="Number of epochs per dataset")
        parser.add_argument('--batch_size', type=int, default=32, help="Batch size for training")
        parser.add_argument('--class_col', type=str, default="SubClass", help="Column name for cell type annotations in .obs")
        parser.add_argument('--true_label_col', type=str, default='cluster_L2', help="Column name for true labels in query .obs for evaluation")

        # --- Evaluation Parameters ---
        parser.add_argument('--emb_keys', nargs='+', default=["X_emb"], help="List of embedding keys in .obsm to use")
        parser.add_argument('--norm_types', nargs='+', default=["raw", "l1", "l2"], help="List of normalization types for embeddings")
        parser.add_argument('--methods', nargs='+', default=["native", "hnsw"], help="List of label transfer methods to test")
        parser.add_argument('--k_values', nargs='+', type=int, default=[30, 50], help="List of K values for KNN-based methods")
        parser.add_argument('--metrics', nargs='+', default=["euclidean", "cosine"], help="List of distance metrics to test")
        parser.add_argument('--unassigned_threshold', type=float, default=None, help="Threshold for marking low-confidence cells as 'Unassigned'")

        # --- Control Flags ---
        parser.add_argument('--force_preprocess', action='store_true', help="Force preprocessing even if cached files exist")
        parser.add_argument('--skip_training', action='store_true', help="Skip model_raw fine-tuning and use existing checkpoint")
        parser.add_argument('--skip_evaluation', action='store_true', help="Skip the evaluation step")
        parser.add_argument('--eval_mode', type=str, default="spatial", choices=["spatial", "benchmark"], help="Choose evaluation mode: 'spatial' or 'benchmark'")

        args = parser.parse_args()
    else:
        args = None  # Debug mode
    main(args)